In [ ]:
# Import libraries for data manipulation, numerical operations, and visualization.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 1. Data loading and variable selection

This project uses selected variables from the 2024 Behavioral Risk Factor Surveillance System (BRFSS) dataset to explore diabetes classification.

I selected variables related to demographic characteristics, BMI, general health, physical activity, smoking, heavy drinking, and reported heart disease. I included both numerical BMI (_BMI5) and BMI category (_BMI5CAT) so I could compare these two representations later.

The analysis focuses on the selected variables rather than loading every column in the original survey dataset.

In [ ]:
# Select the BRFSS variables relevant to this diabetes classification project.
# Both numerical BMI and BMI category are included for a later comparison.


cols = ['DIABETE4','_AGEG5YR', '_SEX', '_BMI5', 'GENHLTH', 'EXERANY2', '_SMOKER3', '_RFDRHV9', 'CVDCRHD4', '_BMI5CAT']

In [ ]:
df = pd.read_csv('brfss_survey_data_2024.csv', usecols=cols)

In [ ]:
# Rename BRFSS variables to descriptive names for easier interpretation.

df = df.rename(columns={
    "DIABETE4": "diabetes_status",
    "_AGEG5YR": "age_group",
    "_SEX": "sex",
    "_BMI5": "bmi",
    "GENHLTH": "general_health",
    "EXERANY2": "exercise_30d",
    "_SMOKER3": "smoking_status",
    "_RFDRHV9": "heavy_drinker",
    "CVDCRHD4": "heart_disease",
    "_BMI5CAT": "bmi_cat"
})
print(df.columns.tolist())

In [ ]:
# Check the number of rows and columns in the selected dataset.

df.shape

In [ ]:
# Inspect the first few records to understand the dataset structure.

df.head()

In [ ]:
# Inspect the last few records as an additional data check.
df.tail()

In [ ]:
# Examine column names, data types, and non-null counts.
# BRFSS variables may be stored as floats even when they represent
# categorical or ordinal responses.

df.info()

In [ ]:
# Check the number of missing values in each variable.

df.isna().sum()

In [ ]:
# BRFSS stores BMI (_BMI5) as an integer representing BMI multiplied by 100. 

print("Before converting")
print(df.bmi.head(), '\n')

# Divide by 100 to convert it to the conventional BMI scale.
df["bmi"] = df["bmi"] / 100

# confirmaing the conversion
print("After converting")
print(df.bmi.head())

In [ ]:
plt.hist(df['bmi'])
plt.show()

In [ ]:
df['bmi'].describe()

### Understanding BRFSS Variable Encodings

The selected BRFSS variables use numeric codes to represent survey responses. I reviewed the coding scheme before cleaning and transforming the variables.

- **general_health:** 1 Excellent, 2 Very good, 3 Good, 4 Fair, 5 Poor, 7 Don’t know, 9 Refused
- **exercise_30d & heart_disease:** 1 Yes, 2 No, 7 Don’t know, 9 Refused
- **sex:** 1 Male, 2 Female
- **age_group:** 1 = 18-24, 2 = 25-29, then every 5 years up to 13 = 80+, 14 = Don’t know/Refused
- **bmi_cat:** 1 Underweight, 2	Normal Weight, 3  Overweight, 4  Obese, Blank is missing
- **smoking_status:** 1 Everyday smoker, 2 Some-days smoker, 3 Former, 4 Never, 9 Don’t know/Refused
- **heavy_drinker:** 1 No, 2 Yes, 9 Don’t know/Refused
- **diabetes_status:** 1 diabetic, 2 only during pregnancy, 3 not diabetic, 4 prediabetes, 7 and 9 (don’t know, refused).

In [ ]:
# Count the number of observation/samples present in each variable
# bmi is skip because it has many observation 

for c in df.columns:
    if c == "bmi":
        continue
    print(df[c].value_counts(dropna=False))
    print()

## 3. Data cleaning and preprocessing
For this initial binary classification task, I retained respondents who reported diabetes (DIABETE4 = 1) and respondents who reported no diabetes (DIABETE4 = 3).

I excluded respondents reporting diabetes only during pregnancy and respondents reporting prediabetes because they do not belong to either target class defined for this model.

I then created a binary target variable:
- 0: respondent reported no diabetes
- 1: respondent reported diabetes
This target represents self-reported diabetes status, not a clinical diagnosis or future diabetes risk.

Some BRFSS variables contain special response codes such as “Don't know” and “Refused.” I removed these codes from the selected predictor variables because they do not represent valid values for the intended analysis.

This filtering approach reduces the number of available observations. The resulting dataset therefore represents respondents with valid answers to the selected variables.

I assigned data types according to the role of each variable. Smoking status and BMI category are treated as categorical features. Age group and general health are retained as ordered numeric variables because their categories have a meaningful order.

This treatment assumes that the ordered categories can be represented numerically for the initial model; it does not imply that the distance between categories is exactly equal.

In [ ]:
# Filtring only the required observation

# Retain respondents who reported diabetes (1) or no diabetes (3).
# Exclude pregnancy-only diabetes and prediabetes because this
# initial model focuses on distinguishing these two groups.

df = df[df["diabetes_status"].isin([1, 3])]

# Convert to integer with 0 for non_diabetic and 1 for diabetic
df["diabetic"] = (df["diabetes_status"] == 1).astype(int)


# Remove responses coded as "Don't know" or "Refused".
# These responses cannot be reliably interpreted as valid
# predictor values for this initial modeling approach.

df = df[df["general_health"].isin([1, 2, 3, 4, 5])]
df = df[df["exercise_30d"].isin([1, 2])]
df = df[df["heart_disease"].isin([1, 2])]
df = df[df["age_group"].between(1, 13)]
df = df[df["smoking_status"].isin([1, 2, 3, 4])]
df = df[df["heavy_drinker"].isin([1, 2])]

# drop columns diabetes_status, since it has been converted to diabetics column
df= df.drop(columns=["diabetes_status"])

print(df.shape)
print(df["diabetic"].value_counts())

In [ ]:
# Recode binary variables into 0/1 indicators, categorical back to object and converting 
# to simplify, model input and interpretation.

# Female = 1, Male = 0
df["sex"] = (df["sex"] == 2).astype(int)

# Exercised in the past 30 days: Yes = 1, No = 0
df["exercise_30d"] =  (df["exercise_30d"] == 1).astype(int)

# Reported heart disease: Yes = 1, No = 0
df["heart_disease"] = (df["heart_disease"] == 1).astype(int)

# Heavy drinking: Yes = 1, No = 0
df["heavy_drinker"] = (df["heavy_drinker"] == 2).astype(int)

# change the data type object since they have multiple categories
for c in ['bmi_cat', 'smoking_status']:
    df[c] = df[c].astype(object)

# float to integer since they are ordinal data 
for i in ["general_health", "age_group"]:
    df[i] = df[i].astype(int)

print(df.dtypes)

### Investigating Missing BMI Values

BMI contains missing values, which may affect the number of
observations available for training.

Before selecting a missing-value strategy, I investigated
whether missing BMI values were distributed differently
across diabetes status and age groups.

In [ ]:
df

In [ ]:
df.isna().sum()

In [ ]:
# checking for correlation 
df.drop(columns=["diabetic"]).corr().round(2)